# RAG ingenuo — Asistente de derecho laboral

SI4006 · Equipo Lawten

Pipeline RAG completo sobre el diccionario de artículos del proyecto: embeddings, recuperación semántica de las tres
referencias más cercanas y la plantilla de respuesta común a todos los sistemas. Se evalúa con el harness del proyecto
(`notebooks/evaluation/harness.py`), el mismo eval set y la misma rúbrica, y se compara con el baseline de M1.

Salidas: `scorecard_rag_ingenuo.csv` (solo las métricas de este sistema) y `scorecard_rag_ingenuo_detalle.csv`.

In [ ]:
import os
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version as version_instalada
from pathlib import Path

from packaging.version import Version

REPO_URL = 'https://github.com/Bosnape/cabrejos-ortiz-valencia-lopez.git'
REPO_BRANCH = 'main'
NOTEBOOK_REL = Path('notebooks/rag/rag_ingenuo.ipynb')

try:
    import google.colab  # type: ignore
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

def buscar_raiz(inicio):
    inicio = Path(inicio).resolve()
    return next((p for p in (inicio, *inicio.parents) if (p / NOTEBOOK_REL).is_file()), None)

RAIZ = buscar_raiz(Path.cwd())
if RAIZ is None and EN_COLAB:
    destino = Path('/content/cabrejos-ortiz-valencia-lopez')
    if not destino.exists():
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(destino)], check=True)
    RAIZ = destino
if RAIZ is None:
    raise RuntimeError('Abra el notebook dentro del repositorio clonado.')
os.chdir(RAIZ)

if EN_COLAB:
    # PEFT rechaza el torchao preinstalado en algunos runtimes; aqui se usa LoRA convencional.
    try:
        torchao = version_instalada('torchao')
    except PackageNotFoundError:
        torchao = None
    if torchao is not None and Version(torchao) < Version('0.16.0'):
        subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
    # Solo lo que este notebook importa; pillow se fija para que ningun paquete lo actualice a mitad de sesion.
    pillow_fija = version_instalada('pillow')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'sentence-transformers==6.0.1',
                    'transformers==5.16.1',
                    'anthropic==1.4.0',
                    'pandas==3.0.5',
                    'python-dotenv==1.2.3',
                    'chromadb', f'pillow=={pillow_fija}'], check=True)

requeridos = [
    Path('notebooks/evaluation/rubrica.json'),
    Path('notebooks/evaluation/eval_set.json'),
    Path('notebooks/evaluation/harness.py'),
    Path('data/diccionario_articulos.csv'),
    Path('notebooks/evaluation/scorecard_baseline.csv'),
]
faltantes = [str(r) for r in requeridos if not (RAIZ / r).is_file()]
if faltantes:
    raise FileNotFoundError('Faltan archivos: ' + ', '.join(faltantes))
sys.path.insert(0, str(RAIZ / 'notebooks/evaluation'))
print('Raiz del proyecto:', RAIZ)

In [ ]:
import json
import random
import time

import numpy as np
import pandas as pd
import torch
from IPython.display import display

import harness as h

EMBED_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
GEN_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
JUEZ_MODEL = 'claude-haiku-4-5'
TOP_K = h.TOP_K
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, torch.cuda.get_device_name(0) if device == 'cuda' else '')

## 1. Corpus

El corpus son los 143 artículos de `data/diccionario_articulos.csv`. Cada artículo es una unidad breve y autocontenida, así
que se indexa completo, sin partirlo en fragmentos.

In [ ]:
eval_set = h.cargar_eval_set(RAIZ)
corpus = h.cargar_corpus(RAIZ)
assert h.validar_eval_set(eval_set, corpus)
print(f'{len(eval_set)} casos; {sum(e["is_adversarial"] for e in eval_set)} adversariales; {len(corpus)} artículos en el corpus.')
display(pd.DataFrame(eval_set).assign(input=lambda d: d['input'].str.slice(0, 90) + '...')[['id', 'case_type', 'is_adversarial', 'input']])

## 2. Recuperación semántica

Los artículos se embeben una sola vez y se indexan en Chroma (distancia coseno). La búsqueda es por similitud contra la
consulta, sin coincidencia de palabras clave; se devuelven las `TOP_K` referencias más cercanas.

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

modelo_embeddings = SentenceTransformer(EMBED_MODEL)

cliente_vectorial = chromadb.Client()
try:
    cliente_vectorial.delete_collection('rag_ingenuo')
except Exception:
    pass
coleccion = cliente_vectorial.create_collection('rag_ingenuo', metadata={'hnsw:space': 'cosine'})
coleccion.add(ids=[str(i) for i in range(len(corpus))], documents=corpus['texto'].tolist(),
              embeddings=modelo_embeddings.encode(corpus['texto'].tolist(), show_progress_bar=False).tolist())

def recuperar(consulta, k=TOP_K):
    r = coleccion.query(query_embeddings=modelo_embeddings.encode([consulta]).tolist(), n_results=k)
    return [int(i) for i in r['ids'][0]]

print('Artículos indexados en Chroma:', coleccion.count())

## 3. Generación

El decoder recibe la consulta y las tres referencias con su descripción, y escribe solo la línea `Pertinentes:` y la
`Valoración:` de la plantilla común. No elige ni cita normas: valora las que recibe.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizador = AutoTokenizer.from_pretrained(GEN_MODEL)
generador = AutoModelForCausalLM.from_pretrained(GEN_MODEL, torch_dtype='auto').to(device).eval()

def generar_texto(sistema, usuario, max_new_tokens):
    chat = [{'role': 'system', 'content': sistema}, {'role': 'user', 'content': usuario}]
    prompt = tokenizador.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    entrada = tokenizador(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        salida = generador.generate(**entrada, max_new_tokens=max_new_tokens, do_sample=False,
                                    pad_token_id=tokenizador.eos_token_id)
    return tokenizador.decode(salida[0][entrada.input_ids.shape[1]:], skip_special_tokens=True).strip()

print('Decoder listo:', GEN_MODEL)

In [ ]:
def sistema(pregunta):
    inicio = time.perf_counter()
    indices = recuperar(pregunta)
    sistema.latencia_recuperacion = time.perf_counter() - inicio
    sistema.consultas_reformuladas = []
    sistema.ultimo_contexto = h.referencias_de(indices, corpus)
    return h.responder(pregunta, sistema.ultimo_contexto, generar_texto)

## 4. Evaluación

Se cargan la rúbrica del proyecto y el juez. Cada respuesta se evalúa en ambos órdenes candidato/referencia para mitigar
el sesgo de posición; el puntaje final es el promedio de los dos órdenes. Las tres dimensiones son las del harness:
recall@3 de la norma esperada, juez y acierto de dominio.

In [ ]:
rubrica_datos, RUBRICA = h.cargar_rubrica(RAIZ)
RUBRICA_VERSION = rubrica_datos['version']
juez = h.crear_juez(RUBRICA, JUEZ_MODEL, RAIZ, EN_COLAB)
print(f'Rúbrica v{RUBRICA_VERSION}')

In [ ]:
METADATOS = {
    'modelo_embeddings': EMBED_MODEL,
    'modelo_generador': GEN_MODEL, 'modelo_juez': JUEZ_MODEL, 'top_k': TOP_K, 'seed': SEED,
    'version_rubrica': RUBRICA_VERSION,
}

resultado = h.harness(eval_set, sistema, juez, metadatos=METADATOS)
print(json.dumps(resultado['resumen'], ensure_ascii=False, indent=2, default=float))
display(pd.DataFrame(resultado['filas']))

## 5. Comparación con el baseline

De `scorecard_baseline.csv` se toman las mismas columnas del scorecard de este sistema. La comparación solo se muestra
aquí; lo que se descarga es el scorecard del RAG ingenuo y su detalle.

In [ ]:
baseline = pd.read_csv(RAIZ / 'notebooks/evaluation/scorecard_baseline.csv')
ingenuo = pd.DataFrame(resultado['filas'])
metricas = ['recall_3', 'juez_mitigado', 'acierto_dominio', 'latencia_seg']
comparacion = baseline[['id'] + metricas].merge(ingenuo[['id'] + metricas], on='id', suffixes=('_baseline', '_ingenuo'))
display(comparacion)
display(pd.DataFrame({'baseline': h.resumir(baseline), 'ingenuo': h.resumir(ingenuo)}).T[
    ['recall_3_medio', 'juez_promedio', 'aciertos', 'latencia_mediana_seg']])

In [ ]:
from pathlib import Path

h.escribir_scorecards(Path('.'), 'rag_ingenuo', resultado)

if EN_COLAB:
    from google.colab import files
    files.download('scorecard_rag_ingenuo.csv')
    files.download('scorecard_rag_ingenuo_detalle.csv')
else:
    print('Guardados ./scorecard_rag_ingenuo.csv y ./scorecard_rag_ingenuo_detalle.csv')

## 6. Interpretación y limitaciones

Se completa con los resultados de la corrida.